In [ ]:
# Fabric pipeline parameter
# "" = run all active SIL_TO_GOLD metadata rows
# Example: "VW_RIDERSHIP_OVERALL" = run only that Gold target

p_target_table = ""


In [ ]:
# =============================================================================
# NB_GOLD_TRANSFORM
# Metadata-driven SILVER / DMA -> GOLD Delta reporting tables
#
# IMPORTANT:
#   - The three GOLD objects are physical DELTA TABLES, not views.
#   - Their names retain the VW_ prefix to match the requested naming.
#   - admin_ops.admin_load_control drives which target executes.
# =============================================================================

import hashlib
import json
import traceback
import uuid
from datetime import datetime, timezone

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType,
)

GOLD_SCHEMA = "gold"
SILVER_SCHEMA = "silver"
DMA_SCHEMA = "dma"
ADMIN_SCHEMA = "admin_ops"

CONTROL_TABLE = "admin_ops.admin_load_control"
PROCESSING_LOG_TABLE = "admin_ops.admin_processing_log"
DQ_TABLE = "admin_ops.admin_dq_results"

EXECUTION_ID = str(uuid.uuid4())
RUN_TS_DT = datetime.now(timezone.utc)
RUN_TS = RUN_TS_DT.strftime("%Y-%m-%d %H:%M:%S")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{GOLD_SCHEMA}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{ADMIN_SCHEMA}`")

for required_table in [
    CONTROL_TABLE,
    PROCESSING_LOG_TABLE,
    DQ_TABLE,
]:
    if not spark.catalog.tableExists(required_table):
        raise ValueError(
            f"Required table {required_table} does not exist. "
            "Create the admin_ops framework tables before running Gold."
        )

print(f"NB_GOLD_TRANSFORM execution_id={EXECUTION_ID}")
print(f"CONTROL_TABLE={CONTROL_TABLE}")
print(f"PROCESSING_LOG_TABLE={PROCESSING_LOG_TABLE}")
print(f"DQ_TABLE={DQ_TABLE}")


In [ ]:
# =============================================================================
# HELPERS: IDs, LOGGING, DQ, VALIDATION, DELTA WRITE
# =============================================================================

def stable_bigint(*parts) -> int:
    payload = "|".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha256(payload.encode("utf-8")).hexdigest()
    return int(digest, 16) & 0x3FFFFFFFFFFFFFFF


def write_processing_log(
    job_name: str,
    job_start_ts: datetime,
    job_status: str,
    source_table: str,
    destination_table: str,
    total_rows: int = 0,
    information: str = None,
    error_id: str = None,
    error_message: str = None,
    processed_condition: str = None,
    flag: str = "SIL_TO_GOLD",
) -> None:
    job_end_ts = datetime.now(timezone.utc)
    duration_sec = int((job_end_ts - job_start_ts).total_seconds())

    row = [(
        stable_bigint(
            EXECUTION_ID,
            job_name,
            source_table,
            job_start_ts.isoformat(),
        ),
        EXECUTION_ID,
        job_name,
        job_start_ts,
        job_end_ts,
        duration_sec,
        job_status,
        information,
        int(total_rows or 0),
        error_id,
        error_message,
        source_table,
        destination_table,
        None,
        None,
        processed_condition,
        flag,
        job_end_ts,
    )]

    schema = StructType([
        StructField("ROW_ID", LongType(), False),
        StructField("EXECUTION_ID", StringType(), False),
        StructField("JOB_NAME", StringType(), False),
        StructField("JOB_START_TS", TimestampType(), False),
        StructField("JOB_END_TS", TimestampType(), False),
        StructField("DURATION_SEC", LongType(), False),
        StructField("JOB_STATUS", StringType(), False),
        StructField("INFORMATION", StringType(), True),
        StructField("TOTAL_ROWS", LongType(), True),
        StructField("ERROR_ID", StringType(), True),
        StructField("ERROR_MESSAGE", StringType(), True),
        StructField("SOURCE_TABLE", StringType(), True),
        StructField("DESTINATION_TABLE", StringType(), True),
        StructField("PULL_FROM_DATE", TimestampType(), True),
        StructField("PULL_TO_DATE", TimestampType(), True),
        StructField("PROCESSED_CONDITION", StringType(), True),
        StructField("FLAG", StringType(), True),
        StructField("INSERT_TS", TimestampType(), False),
    ])

    (
        spark.createDataFrame(row, schema=schema)
        .write
        .mode("append")
        .saveAsTable(PROCESSING_LOG_TABLE)
    )


def log_dq(
    target_table: str,
    source_tables: str,
    rule_name: str,
    rule_type: str,
    description: str,
    severity: str,
    total_rows: int,
    failed_rows: int,
    sample_details: str = None,
) -> None:
    check_ts = datetime.now(timezone.utc)
    total_rows = int(total_rows or 0)
    failed_rows = int(failed_rows or 0)

    failed_pct = (
        round((failed_rows / total_rows) * 100.0, 6)
        if total_rows > 0
        else 0.0
    )

    status = "PASS" if failed_rows == 0 else "FAIL"

    row = [(
        stable_bigint(
            EXECUTION_ID,
            target_table,
            rule_name,
            check_ts.isoformat(),
        ),
        EXECUTION_ID,
        "GOLD",
        target_table,
        source_tables,
        rule_name,
        rule_type,
        description,
        severity.upper(),
        status,
        total_rows,
        failed_rows,
        failed_pct,
        sample_details,
        check_ts,
        check_ts,
    )]

    schema = StructType([
        StructField("DQ_RESULT_ID", LongType(), False),
        StructField("EXECUTION_ID", StringType(), False),
        StructField("LAYER_NAME", StringType(), False),
        StructField("TARGET_TABLE", StringType(), False),
        StructField("SOURCE_TABLES", StringType(), True),
        StructField("RULE_NAME", StringType(), False),
        StructField("RULE_TYPE", StringType(), False),
        StructField("RULE_DESCRIPTION", StringType(), True),
        StructField("SEVERITY", StringType(), False),
        StructField("RULE_STATUS", StringType(), False),
        StructField("TOTAL_ROW_COUNT", LongType(), False),
        StructField("FAILED_ROW_COUNT", LongType(), False),
        StructField("FAILED_PCT", DoubleType(), False),
        StructField("SAMPLE_DETAILS", StringType(), True),
        StructField("CHECK_TS", TimestampType(), False),
        StructField("INSERT_TS", TimestampType(), False),
    ])

    (
        spark.createDataFrame(row, schema=schema)
        .write
        .mode("append")
        .saveAsTable(DQ_TABLE)
    )


def run_dq(
    df: DataFrame,
    target_table: str,
    source_tables: str,
    rule_name: str,
    rule_type: str,
    description: str,
    severity: str,
    fail_condition,
    sample_columns=None,
    sample_limit: int = 5,
) -> int:
    total_rows = df.count()
    failed_df = df.filter(fail_condition)
    failed_rows = failed_df.count()

    sample_details = None

    if failed_rows > 0:
        cols = sample_columns or df.columns[:8]
        samples = [
            r.asDict(recursive=True)
            for r in failed_df.select(*cols).limit(sample_limit).collect()
        ]
        sample_details = json.dumps(samples, default=str)[:4000]

    log_dq(
        target_table=target_table,
        source_tables=source_tables,
        rule_name=rule_name,
        rule_type=rule_type,
        description=description,
        severity=severity,
        total_rows=total_rows,
        failed_rows=failed_rows,
        sample_details=sample_details,
    )

    if failed_rows > 0 and severity.upper() == "ERROR":
        raise ValueError(
            f"DQ ERROR [{target_table}.{rule_name}] "
            f"{failed_rows:,}/{total_rows:,} row(s) failed"
        )

    if failed_rows > 0:
        print(
            f"DQ WARNING [{target_table}.{rule_name}] "
            f"{failed_rows:,}/{total_rows:,} row(s) failed"
        )

    return failed_rows


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required source table {table_name} does not exist."
        )


def require_columns(
    df: DataFrame,
    table_name: str,
    required_columns,
) -> None:
    existing = {c.upper() for c in df.columns}
    missing = [
        c for c in required_columns
        if c.upper() not in existing
    ]

    if missing:
        raise ValueError(
            f"{table_name} is missing required column(s): {missing}. "
            f"Available columns: {df.columns}"
        )


def parse_date_expression(column_name: str):
    c = F.col(column_name)

    return F.coalesce(
        c.cast("date"),
        F.to_date(c.cast("string"), "yyyy-MM-dd"),
        F.to_date(c.cast("string"), "MM/dd/yyyy"),
        F.to_date(c.cast("string"), "dd-MM-yyyy"),
        F.to_date(c.cast("string")),
    )


def overwrite_gold_table(
    df: DataFrame,
    target_table: str,
) -> int:
    qualified_target = f"{GOLD_SCHEMA}.{target_table}"

    row_count = df.count()

    if row_count == 0:
        raise ValueError(
            f"{qualified_target}: source transformation returned zero rows. "
            "Gold table was not overwritten."
        )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(qualified_target)
    )

    print(
        f"{qualified_target}: overwritten as Delta table "
        f"with {row_count:,} row(s)"
    )

    return row_count


In [ ]:
# =============================================================================
# GOLD TABLE 1: VW_SCHEDULED_RIDERSHIP
# Physical Delta table despite VW_ prefix.
# =============================================================================

def create_vw_scheduled_ridership(
    target_table="VW_SCHEDULED_RIDERSHIP",
):
    source = f"{SILVER_SCHEMA}.SILVER_FACT_RIDERSHIP"
    require_table(source)

    # No revised scheduled-ridership projection was supplied separately.
    # Keep the full Silver fact as the scheduled Gold reporting table.
    df = spark.table(source)

    run_dq(
        df,
        target_table,
        source,
        "SERVICE_DATE_NOT_NULL",
        "NOT_NULL",
        "Scheduled ridership SERVICE_DATE must not be null",
        "ERROR",
        F.col("SERVICE_DATE").isNull(),
        ["TRIP_ID", "SERVICE_DATE"],
    )

    run_dq(
        df,
        target_table,
        source,
        "PSGR_ON_COUNT_NON_NEGATIVE",
        "BUSINESS_RULE",
        "Scheduled ridership passenger count must not be negative",
        "ERROR",
        F.col("PSGR_ON_COUNT") < 0,
        ["TRIP_ID", "PSGR_ON_COUNT", "SERVICE_DATE"],
    )

    return overwrite_gold_table(
        df,
        target_table,
    )


# =============================================================================
# GOLD TABLE 2: VW_RIDERSHIP_OVERALL
# Scheduled Silver fact + DMA unscheduled trips
# =============================================================================

def create_vw_ridership_overall(
    target_table="VW_RIDERSHIP_OVERALL",
):
    scheduled_table = (
        f"{SILVER_SCHEMA}.SILVER_FACT_RIDERSHIP"
    )

    unscheduled_table = (
        f"{DMA_SCHEMA}.unscheduled_trip"
    )

    require_table(scheduled_table)
    require_table(unscheduled_table)

    scheduled_df = spark.table(
        scheduled_table
    )

    require_columns(
        scheduled_df,
        scheduled_table,
        [
            "SERVICE_DATE",
            "TRIP_ID",
            "ROUTE_ID",
            "DIRECTION_ID",
            "VEHICLE_ID",
            "BLOCK_ID",
            "PSGR_ON_COUNT",
        ],
    )

    scheduled = (
        scheduled_df
        .select(
            F.lit("Scheduled").alias("TRIP_TYPE"),
            F.col("SERVICE_DATE").cast("date").alias("SERVICE_DATE"),
            F.col("TRIP_ID").cast("long").alias("TRIP_ID"),
            F.col("ROUTE_ID").cast("long").alias("ROUTE_ID"),
            F.col("DIRECTION_ID").cast("long").alias("DIRECTION_ID"),
            F.col("VEHICLE_ID").cast("long").alias("VEHICLE_ID"),
            F.col("BLOCK_ID").cast("long").alias("BLOCK_ID"),
            F.col("PSGR_ON_COUNT").cast("long").alias("PSGR_ON_COUNT"),
        )
    )

    unscheduled_df = spark.table(
        unscheduled_table
    )

    require_columns(
        unscheduled_df,
        unscheduled_table,
        [
            "TRIP_DATE",
            "ROUTE_ID",
            "DIRECTION_ID",
            "VEHICLE_ID",
            "BLOCK_ID",
            "RIDERSHIP",
        ],
    )

    # Legacy Snowflake definition expects UNSCHEDULED_TRIP_ID.
    # If the current DMA Excel load already contains it, preserve it.
    # If it does not, create a deterministic ID from the available
    # unscheduled-trip business fields so Gold can retain the expected grain.
    unscheduled_columns = {
        c.upper(): c
        for c in unscheduled_df.columns
    }

    if "UNSCHEDULED_TRIP_ID" not in unscheduled_columns:
        print(
            "dma.unscheduled_trip does not contain UNSCHEDULED_TRIP_ID; "
            "creating a deterministic ID for the Gold table."
        )

        preferred_key_columns = [
            "TRIP_DATE",
            "ROUTE_ID",
            "DIRECTION_ID",
            "VEHICLE_ID",
            "BLOCK_ID",
            "START_TIME",
            "END_TIME",
            "TRIP_TYPE",
            "TRIP_DESC",
            "RIDERSHIP",
        ]

        available_key_columns = [
            unscheduled_columns[c]
            for c in preferred_key_columns
            if c in unscheduled_columns
        ]

        hash_terms = [
            F.coalesce(
                F.col(c).cast("string"),
                F.lit("__NULL__"),
            )
            for c in available_key_columns
        ]

        unscheduled_df = (
            unscheduled_df
            .withColumn(
                "UNSCHEDULED_TRIP_ID",
                F.xxhash64(*hash_terms),
            )
        )

    unscheduled = (
        unscheduled_df
        .select(
            F.lit("Unscheduled").alias("TRIP_TYPE"),
            parse_date_expression("TRIP_DATE").alias("SERVICE_DATE"),
            F.col("UNSCHEDULED_TRIP_ID").cast("long").alias("TRIP_ID"),
            F.col("ROUTE_ID").cast("long").alias("ROUTE_ID"),
            F.col("DIRECTION_ID").cast("long").alias("DIRECTION_ID"),
            F.col("VEHICLE_ID").cast("long").alias("VEHICLE_ID"),
            F.col("BLOCK_ID").cast("long").alias("BLOCK_ID"),
            F.coalesce(
                F.col("RIDERSHIP").cast("long"),
                F.lit(0).cast("long"),
            ).alias("PSGR_ON_COUNT"),
        )
    )

    df = scheduled.unionByName(
        unscheduled
    )

    source = (
        f"{scheduled_table},"
        f"{unscheduled_table}"
    )

    run_dq(
        df,
        target_table,
        source,
        "SERVICE_DATE_NOT_NULL",
        "NOT_NULL",
        "Overall ridership SERVICE_DATE must not be null",
        "ERROR",
        F.col("SERVICE_DATE").isNull(),
        ["TRIP_TYPE", "TRIP_ID", "SERVICE_DATE"],
    )

    run_dq(
        df,
        target_table,
        source,
        "TRIP_TYPE_VALID",
        "DOMAIN",
        "TRIP_TYPE must be Scheduled or Unscheduled",
        "ERROR",
        ~F.col("TRIP_TYPE").isin(
            "Scheduled",
            "Unscheduled",
        ),
        ["TRIP_TYPE", "TRIP_ID"],
    )

    run_dq(
        df,
        target_table,
        source,
        "PSGR_ON_COUNT_NON_NEGATIVE",
        "BUSINESS_RULE",
        "Overall ridership passenger count must not be negative",
        "ERROR",
        F.col("PSGR_ON_COUNT") < 0,
        [
            "TRIP_TYPE",
            "TRIP_ID",
            "PSGR_ON_COUNT",
            "SERVICE_DATE",
        ],
    )

    return overwrite_gold_table(
        df,
        target_table,
    )


# =============================================================================
# GOLD TABLE 3: VW_RIDERSHIP_DETAIL
# Fabric / Spark SQL version of the supplied Gold definition.
# =============================================================================

def create_vw_ridership_detail(
    target_table="VW_RIDERSHIP_DETAIL",
):
    required_tables = [
        f"{SILVER_SCHEMA}.SILVER_FACT_RIDERSHIP",
        f"{SILVER_SCHEMA}.SILVER_DIM_ROUTE",
        f"{SILVER_SCHEMA}.SILVER_DIM_TRIP",
        f"{SILVER_SCHEMA}.SILVER_DIM_DATE",
        f"{SILVER_SCHEMA}.SILVER_DIM_DIRECTION",
        f"{SILVER_SCHEMA}.SILVER_DIM_SCHEDULE",
        f"{SILVER_SCHEMA}.SILVER_DIM_STOP",
        f"{SILVER_SCHEMA}.SILVER_DIM_VEHICLE",
    ]

    for table_name in required_tables:
        require_table(table_name)

    df = spark.sql(
        f"""
        WITH base AS
        (
            SELECT
                FR.STOP_ID,
                FR.SCHEDULE_ID,
                FR.BLOCK_ID,
                FR.VEHICLE_ID,
                FR.TRIP_ID,
                FR.RIDERSHIP_VALIDATION_ID,
                FR.ROUTE_ID,
                FR.DIRECTION_ID,
                FR.DIR_ROUTE_SCHEDULE_ID,

                COALESCE(
                    FR.ROUTE_SCHEDULE_ID,
                    DR.ROUTE_SCHEDULE_ID
                ) AS ROUTE_SCHEDULE_ID,

                FR.SERVICE_DATE,

                FR.LATITUDE,
                FR.LONGITUDE,

                FR.PSGR_ON_COUNT,
                FR.PSGR_MILES,
                FR.PSGR_HRS,

                DR.ROUTE_NAME,
                DR.ROUTE_TYPE,

                DT.IS_REVENUE_TRIP,
                DT.TRIP_START_TIME,

                CAST(
                    SPLIT(
                        TRIM(DT.TRIP_START_TIME),
                        ':'
                    )[0]
                    AS INT
                ) AS TRIP_START_HOUR,

                DT.TRIP_END_TIME,

                INITCAP(
                    REPLACE(
                        DT.TRIP_TYPE,
                        '_',
                        ' '
                    )
                ) AS TRIP_TYPE,

                DD.DATE_ID,

                CASE
                    WHEN UPPER(TRIM(DD.SERVICE_DAY)) = 'SATURDAY'
                        THEN 'Saturday'
                    WHEN UPPER(TRIM(DD.SERVICE_DAY)) = 'SUNDAY'
                        THEN 'Sunday'
                    ELSE 'Weekday'
                END AS SERVICE_DAY_TYPE,

                DD.MONTH_NAME,
                DD.CALENDER_YEAR,
                DD.CALENDER_MONTH_NUM,
                DD.MONTH_NAME_YEAR,
                DD.WEEK_NUM,

                DDR.DIRECTION_ABBR,

                INITCAP(
                    DDR.DIRECTION_NAME
                ) AS DIRECTION_NAME,

                DS.SCHEDULE_DESC,
                DS.SCHEDULE_START_DATE,
                DS.SCHEDULE_END_DATE,

                DST.STOP_ABBR,
                DST.STOP_NAME,
                DST.CITY,
                DST.ZIPCODE,
                DST.ZONE,
                DST.STOP_TYPE,
                DST.IS_WHEELCHAIR_ACCESSIBLE,

                DV.VEHICLE_MAKE,
                DV.VEHICLE_MODEL,
                DV.VEHICLE_TYPE_DESC,
                DV.VEHICLE_PSGR_CAPACITY,
                DV.VEHICLE_YEAR,
                DV.WHEELCHAIR_CAPACITY

            FROM {SILVER_SCHEMA}.SILVER_FACT_RIDERSHIP FR

            LEFT JOIN {SILVER_SCHEMA}.SILVER_DIM_ROUTE DR
                ON DR.ROUTE_SCHEDULE_ID
                 = FR.ROUTE_SCHEDULE_ID

            LEFT JOIN {SILVER_SCHEMA}.SILVER_DIM_TRIP DT
                ON DT.TRIP_ID
                 = FR.TRIP_ID

            LEFT JOIN {SILVER_SCHEMA}.SILVER_DIM_DATE DD
                ON DD.SERVICE_DATE
                 = FR.SERVICE_DATE

            LEFT JOIN {SILVER_SCHEMA}.SILVER_DIM_DIRECTION DDR
                ON DDR.DIRECTION_ID
                 = FR.DIRECTION_ID

            LEFT JOIN {SILVER_SCHEMA}.SILVER_DIM_SCHEDULE DS
                ON DS.SCHEDULE_ID
                 = FR.SCHEDULE_ID

            LEFT JOIN {SILVER_SCHEMA}.SILVER_DIM_STOP DST
                ON DST.STOP_ID
                 = FR.STOP_ID

            LEFT JOIN {SILVER_SCHEMA}.SILVER_DIM_VEHICLE DV
                ON DV.VEHICLE_ID
                 = FR.VEHICLE_ID
        )

        SELECT
            STOP_ID,
            SCHEDULE_ID,
            BLOCK_ID,
            VEHICLE_ID,
            TRIP_ID,
            RIDERSHIP_VALIDATION_ID,
            ROUTE_ID,
            DIRECTION_ID,
            DIR_ROUTE_SCHEDULE_ID,
            ROUTE_SCHEDULE_ID,
            SERVICE_DATE,
            LATITUDE,
            LONGITUDE,
            PSGR_ON_COUNT,
            PSGR_MILES,
            PSGR_HRS,
            ROUTE_NAME,
            ROUTE_TYPE,
            IS_REVENUE_TRIP,
            TRIP_START_TIME,
            TRIP_START_HOUR,

            CASE
                WHEN TRIP_START_HOUR >= 23
                  OR TRIP_START_HOUR <= 4
                    THEN 'Late Night'
                WHEN TRIP_START_HOUR < 9
                    THEN 'Early Morning'
                WHEN TRIP_START_HOUR < 12
                    THEN 'Morning'
                WHEN TRIP_START_HOUR < 15
                    THEN 'Afternoon'
                WHEN TRIP_START_HOUR < 19
                    THEN 'Evening Peak'
                WHEN TRIP_START_HOUR <= 22
                    THEN 'Evening'
            END AS TRIP_HOUR_BUCKET,

            CASE
                WHEN TRIP_START_HOUR >= 23
                  OR TRIP_START_HOUR <= 4
                    THEN 6
                WHEN TRIP_START_HOUR < 9
                    THEN 1
                WHEN TRIP_START_HOUR < 12
                    THEN 2
                WHEN TRIP_START_HOUR < 15
                    THEN 3
                WHEN TRIP_START_HOUR < 19
                    THEN 4
                WHEN TRIP_START_HOUR <= 22
                    THEN 5
            END AS TRIP_HOUR_BUCKET_SORT,

            TRIP_END_TIME,
            TRIP_TYPE,
            DATE_ID,
            SERVICE_DAY_TYPE,

            CASE
                WHEN SERVICE_DAY_TYPE = 'Weekday'
                    THEN 1
                WHEN SERVICE_DAY_TYPE = 'Saturday'
                    THEN 2
                WHEN SERVICE_DAY_TYPE = 'Sunday'
                    THEN 3
                ELSE 4
            END AS SERVICE_DAY_TYPE_SORT,

            MONTH_NAME,
            CALENDER_YEAR,
            CALENDER_MONTH_NUM,
            MONTH_NAME_YEAR,
            WEEK_NUM,
            DIRECTION_ABBR,
            DIRECTION_NAME,
            SCHEDULE_DESC,
            SCHEDULE_START_DATE,
            SCHEDULE_END_DATE,
            STOP_ABBR,
            STOP_NAME,
            CITY,
            ZIPCODE,
            ZONE,
            STOP_TYPE,
            IS_WHEELCHAIR_ACCESSIBLE,
            VEHICLE_MAKE,
            VEHICLE_MODEL,
            VEHICLE_TYPE_DESC,
            VEHICLE_PSGR_CAPACITY,
            VEHICLE_YEAR,
            WHEELCHAIR_CAPACITY

        FROM base
        """
    )

    source = ",".join(
        required_tables
    )

    run_dq(
        df,
        target_table,
        source,
        "SERVICE_DATE_NOT_NULL",
        "NOT_NULL",
        "Ridership detail SERVICE_DATE must not be null",
        "ERROR",
        F.col("SERVICE_DATE").isNull(),
        ["TRIP_ID", "STOP_ID", "SERVICE_DATE"],
    )

    run_dq(
        df,
        target_table,
        source,
        "PSGR_ON_COUNT_NON_NEGATIVE",
        "BUSINESS_RULE",
        "Ridership detail passenger count must not be negative",
        "ERROR",
        F.col("PSGR_ON_COUNT") < 0,
        [
            "TRIP_ID",
            "STOP_ID",
            "PSGR_ON_COUNT",
            "SERVICE_DATE",
        ],
    )

    return overwrite_gold_table(
        df,
        target_table,
    )


In [ ]:
# =============================================================================
# METADATA-DRIVEN GOLD DRIVER
# =============================================================================

LOADERS = {
    "create_vw_scheduled_ridership":
        create_vw_scheduled_ridership,

    "create_vw_ridership_overall":
        create_vw_ridership_overall,

    "create_vw_ridership_detail":
        create_vw_ridership_detail,
}


p_target_table_clean = (
    p_target_table
    or ""
).strip()

p_target_table_escaped = (
    p_target_table_clean
    .replace("'", "''")
)

target_filter_clause = ""

if p_target_table_clean:
    target_filter_clause = f"""
      AND (
            LOWER(TRIM(DESTINATION_TABLE_NAME))
                = LOWER(TRIM('{p_target_table_escaped}'))

         OR LOWER(TRIM(PARAM_1))
                = LOWER(TRIM('{p_target_table_escaped}'))

         OR TRIM(CAST(ROW_ID AS STRING))
                = TRIM('{p_target_table_escaped}')
      )
    """


control_df = spark.sql(
    f"""
    SELECT
        ROW_ID,
        TRANSFER_TYPE,
        SOURCE_SYSTEM,
        SOURCE_SCHEMA,
        SOURCE_TABLE_NAME,
        DESTINATION_SCHEMA,
        DESTINATION_TABLE_NAME,
        LOAD_TYPE,
        EXECUTION_ORDER,
        JOB_DEPENDENCIES,
        RUN_ACTIVE_FLAG,
        PARAM_1,
        PARAM_2,
        INSERT_TS,
        UPDATE_TS
    FROM {CONTROL_TABLE}
    WHERE RUN_ACTIVE_FLAG = 1
      AND TRANSFER_TYPE = 'SIL_TO_GOLD'
      {target_filter_clause}
    ORDER BY EXECUTION_ORDER, ROW_ID
    """
)


control_rows = control_df.collect()

print(
    f"Gold p_target_table="
    f"{p_target_table_clean or '<ALL>'}"
)

print(
    f"Active Gold jobs found: "
    f"{len(control_rows)}"
)

display(
    control_df
)


if not control_rows:
    raise ValueError(
        f"No active SIL_TO_GOLD rows found in {CONTROL_TABLE} "
        f"for p_target_table={p_target_table_clean or '<ALL>'}."
    )


results = []


for row in control_rows:
    fields = row.asDict(recursive=True)

    row_id = fields.get("ROW_ID")
    loader_name = (
        fields.get("PARAM_1")
        or ""
    ).strip()

    source_tables = (
        fields.get("SOURCE_TABLE_NAME")
        or ""
    ).strip()

    destination_schema = (
        fields.get("DESTINATION_SCHEMA")
        or GOLD_SCHEMA
    ).strip().lower()

    destination_table = (
        fields.get("DESTINATION_TABLE_NAME")
        or ""
    ).strip()

    load_type = (
        fields.get("LOAD_TYPE")
        or "TABLE"
    ).strip().upper()

    job_name = destination_table
    job_start_ts = datetime.now(timezone.utc)

    print("=" * 90)
    print(f"ROW ID       : {row_id}")
    print(f"GOLD JOB     : {job_name}")
    print(f"LOADER       : {loader_name}")
    print(f"SOURCES      : {source_tables}")
    print(
        f"DESTINATION  : "
        f"{destination_schema}.{destination_table}"
    )
    print(f"LOAD TYPE    : {load_type}")
    print("=" * 90)

    try:
        if destination_schema != GOLD_SCHEMA:
            raise ValueError(
                f"{job_name}: DESTINATION_SCHEMA must be "
                f"'{GOLD_SCHEMA}', received '{destination_schema}'."
            )

        if load_type != "TABLE":
            raise ValueError(
                f"{job_name}: LOAD_TYPE must be TABLE because Gold "
                f"objects are physical Delta tables. Received {load_type}."
            )

        if not destination_table:
            raise ValueError(
                f"Gold metadata ROW_ID={row_id} has no "
                "DESTINATION_TABLE_NAME."
            )

        if loader_name not in LOADERS:
            raise ValueError(
                f"{job_name}: unknown Gold loader PARAM_1="
                f"'{loader_name}'. Expected one of "
                f"{sorted(LOADERS.keys())}."
            )

        loader = LOADERS[
            loader_name
        ]

        row_count = loader(
            destination_table
        )

        write_processing_log(
            job_name=job_name,
            job_start_ts=job_start_ts,
            job_status="SUCCESS",
            source_table=source_tables,
            destination_table=(
                f"{destination_schema}."
                f"{destination_table}"
            ),
            total_rows=row_count,
            information=(
                f"Gold Delta table overwritten successfully; "
                f"loader={loader_name}; "
                f"rows={row_count}; "
                f"load_type={load_type}"
            ),
            flag="SIL_TO_GOLD",
        )

        result = {
            "row_id": row_id,
            "job": job_name,
            "loader": loader_name,
            "status": "SUCCESS",
            "rows": row_count,
        }

        print(
            f"SUCCESS: {job_name} "
            f"({row_count:,} rows)"
        )

    except Exception as exc:
        error_id = str(
            uuid.uuid4()
        )

        error_text = (
            f"{exc}\n"
            f"{traceback.format_exc()}"
        )

        try:
            write_processing_log(
                job_name=(
                    job_name
                    or f"ROW_ID_{row_id}"
                ),
                job_start_ts=job_start_ts,
                job_status="FAILED",
                source_table=source_tables,
                destination_table=(
                    f"{destination_schema}."
                    f"{destination_table}"
                    if destination_table
                    else destination_schema
                ),
                total_rows=0,
                information=(
                    f"Gold loader failed; "
                    f"loader={loader_name}; "
                    f"load_type={load_type}"
                ),
                error_id=error_id,
                error_message=(
                    error_text[:8000]
                ),
                flag="SIL_TO_GOLD",
            )

        except Exception as log_exc:
            print(
                f"WARNING: could not write failure log "
                f"for {job_name}: {log_exc}"
            )

        result = {
            "row_id": row_id,
            "job": (
                job_name
                or f"ROW_ID_{row_id}"
            ),
            "loader": loader_name,
            "status": "FAILED",
            "rows": 0,
            "error": str(exc),
        }

        print(
            f"FAILED: "
            f"{result['job']} - {exc}"
        )

    results.append(
        result
    )


successes = [
    r for r in results
    if r["status"] == "SUCCESS"
]

failures = [
    r for r in results
    if r["status"] == "FAILED"
]


print("=" * 90)

print(
    "Gold run complete. "
    f"Success={len(successes)} "
    f"Failed={len(failures)}"
)

for result in results:
    print(result)


if failures:
    failed_jobs = ", ".join(
        r["job"]
        for r in failures
    )

    raise RuntimeError(
        f"Gold transformation failed for: "
        f"{failed_jobs}"
    )


In [ ]:
%%sql

-- ============================================================
-- 1. SOURCE COUNTS
-- ============================================================

SELECT
    'SILVER_FACT_RIDERSHIP' AS TABLE_NAME,
    COUNT(*) AS ROW_COUNT
FROM silver.SILVER_FACT_RIDERSHIP

UNION ALL

SELECT
    'DMA_UNSCHEDULED_TRIP',
    COUNT(*)
FROM dma.unscheduled_trip

UNION ALL

SELECT
    'GOLD_VW_SCHEDULED_RIDERSHIP',
    COUNT(*)
FROM gold.VW_SCHEDULED_RIDERSHIP

UNION ALL

SELECT
    'GOLD_VW_RIDERSHIP_OVERALL',
    COUNT(*)
FROM gold.VW_RIDERSHIP_OVERALL

UNION ALL

SELECT
    'GOLD_VW_RIDERSHIP_DETAIL',
    COUNT(*)
FROM gold.VW_RIDERSHIP_DETAIL;

